In [1]:
from transformers import AutoProcessor
from transformers import SeamlessM4TModel
import torch
import pandas as pd

tgt_langs = ['Chinese']
from multilingualmc.translator.get_terms import TermCollector
term_collector = TermCollector('/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_final', tgt_langs)

processor = AutoProcessor.from_pretrained("facebook/hf-seamless-m4t-Large", use_fast=False, force_download=True, cache_dir='/data/user_data/jiaruil5/.cache/')
model = SeamlessM4TModel.from_pretrained("facebook/hf-seamless-m4t-Large", cache_dir='/data/user_data/jiaruil5/.cache/')

/data/user_data/jiaruil5/miniconda3/envs/mmc/lib/python3.10/site-packages/transformers/deepspeed.py:23: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(
/data/user_data/jiaruil5/miniconda3/envs/mmc/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/data/user_data/jiaruil5/miniconda3/envs/mmc/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [2]:
from transformers import LogitsProcessor

seamless_lang_dict = {
    "Arabic": "arb",
    "Chinese": "cmn",
    "English": "eng",
    "French": "fra",
    "Japanese": "jpn",
    "Russian": "rus",
}

class TerminologyAwareLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, en_text, lang, soft_penalty, seamless_lang_dict, tokens):
        self.tokenizer = tokenizer
        self.en_text = en_text
        self.lang = lang
        self.soft_penalty = soft_penalty
        self.seamless_lang_dict = seamless_lang_dict
        self.tokens = tokens

    def __call__(self, input_ids, scores):
        # print(scores)
        for idx in range(scores.shape[-1]):
            tokens = list({item for sublist in self.tokens for item in sublist})
            if idx in tokens:  # Penalize tokens not in the translation
                scores[:, idx] /= self.soft_penalty  # Apply soft penalty
        # print(scores)
        return scores

def generate_with_constraints(model, tokenizer, ai_terms_dict, en_text, lang, num_beams=5, soft_penalty=0.8):
    # Tokenize the input text
    inputs = tokenizer(en_text, src_lang = seamless_lang_dict['English'], return_tensors="pt")
    # print(inputs)
    
    tokens = []
    for translation in ai_terms_dict:
        token = tokenizer(translation, src_lang=seamless_lang_dict[lang])['input_ids']
        tokens.append(token)
    # print(tokens)
    # print(terms)
    
    logits_processor = TerminologyAwareLogitsProcessor(
        tokenizer=tokenizer,
        en_text=en_text,
        lang=lang,
        soft_penalty=soft_penalty,
        seamless_lang_dict=seamless_lang_dict,
        tokens = tokens,
    )

    # Prepare the beam search decoder
    generated_ids = model.generate(
        **inputs,
        tgt_lang=seamless_lang_dict[lang],
        generate_speech=False,
        # num_beams=num_beams,
        # early_stopping=True,
        # output_scores=True,
        # return_dict_in_generate=True,
        logits_processor=[logits_processor]
    )

    # Decode the output sequence
    output = tokenizer.decode(generated_ids[0].tolist()[0], skip_special_tokens=True)
    return output

In [3]:

input_text = "1: For all the methods, we used 10-fold cross validation (i.e., each fold we have 556 training and 62 test samples) to tune free parameters, e.g., the kernel form and parameters for GPOR and LapSVM. Note that all the alternative methods stack X and Z together into a whole data matrix and ignore their heterogeneous nature."

force_words = list(set([term_collector.terms_dict[key]['Chinese'] for key in term_collector.find_terminology(input_text)]))
force_words

['训练', '核函数', '矩阵', '10折交叉验证']

In [8]:
translation = generate_with_constraints(model, processor, force_words, input_text, lang='Chinese', soft_penalty=0.7)
print("Translated Text:", translation)

Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.


Translated Text: 1:对于所有方法,我们使用 10 倍交叉验证 (即,每折我们有 556 训练和 62 测试样本) 调整自由参数,例如核形式和 GPOR 和 LapSVM 参数.


In [7]:
text_inputs = processor(
    text=input_text,
    src_lang=seamless_lang_dict['English'],
    return_tensors='pt'
)
output_tokens = model.generate(**text_inputs, tgt_lang=seamless_lang_dict['Chinese'], generate_speech=False)
processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

Keyword arguments {'add_special_tokens': False} not recognized.


'1:对于所有方法,我们使用了10倍的交叉验证(即,每倍我们有556个训练和62个测试样本)来调整自由参数,例如,内核形式和GPOR和LapSVM的参数.'